In [ ]:
import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

def train_combined_fuel_model():
    # Initialize empty list to store dataframes
    all_data = []
    
    # Read and combine all fuel type data
    fuel_types = ['ethanol', 'petrol', 'diesel']
    for fuel_type in fuel_types:
        data = pd.read_csv(f'../datasets/{fuel_type}.csv')
        data['fuel_type'] = fuel_type  # Add fuel type as a feature
        all_data.append(data)
    
    # Combine all datasets
    combined_data = pd.concat(all_data, ignore_index=True)
    
    # Convert 'created_at' to datetime and extract features
    combined_data['created_at'] = pd.to_datetime(combined_data['created_at'])
    combined_data['month'] = combined_data['created_at'].dt.month
    combined_data['day_of_week'] = combined_data['created_at'].dt.dayofweek
    combined_data['year'] = combined_data['created_at'].dt.year
    combined_data['days_since_start'] = (combined_data['created_at'] - combined_data['created_at'].min()).dt.days
    
    # Convert fuel type to categorical
    combined_data = pd.get_dummies(combined_data, columns=['fuel_type'])
    
    # Define features and target
    features = ['volume', 'month', 'day_of_week', 'year', 'days_since_start', 
                'fuel_type_ethanol', 'fuel_type_petrol', 'fuel_type_diesel','exchange_rate','inflation_rate','crude_oil_price','demand_index']
    X = combined_data[features]
    y = combined_data['price']
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    
    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Evaluate
    y_predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, y_predictions)
    print(f'Mean Squared Error for combined model: {mse}')
    
    # Save model
    model_filename = '../models/fuel_price_model.joblib'
    joblib.dump(model, model_filename)
    print(f"Model saved to {model_filename}")
    
    return model, mse

# Train combined model
print("\nTraining combined fuel model...")
model, score = train_combined_fuel_model()



Training combined fuel model...
Mean Squared Error for combined model: 0.32859388181996535
Model saved to ../models/fuel_price_model.joblib
